In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install pandas xarray pyarrow fastparquet plotly

import os
import pandas as pd
import plotly.express as px

# --- STEP 1: SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")

# Check available filtered files
filtered_files = [f for f in os.listdir(filtered_dir) if f.endswith(".csv")]
print("📂 Filtered files available:", filtered_files)

# --- STEP 2: FUNCTION TO LOAD AND STANDARDIZE DATA ---
def load_and_standardize(file_path):
    """
    Load filtered CSV and standardize column names to:
    LATITUDE, LONGITUDE, JULD, DEPTH_M, TEMP, PSAL
    """
    df = pd.read_csv(file_path)
    # Rename columns if needed
    rename_map = {
        "LAT": "LATITUDE",
        "LON": "LONGITUDE",
        "TEMP_ADJUSTED": "TEMP",
        "DEPTH": "DEPTH_M"
    }
    df.rename(columns=rename_map, inplace=True)

    # Ensure required columns exist
    required_cols = ["LATITUDE", "LONGITUDE", "JULD", "DEPTH_M", "TEMP", "PSAL"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = pd.NA  # fill missing columns with NA

    return df[required_cols + ["source_file"]]

# --- STEP 3: FUNCTION TO PLOT VARIABLE ---
def plot_variable(df, var="TEMP", depth_col="DEPTH_M", interactive=True):
    if df.empty or var not in df.columns or depth_col not in df.columns:
        print(f"⚠️ Columns {var} or {depth_col} not found in dataframe")
        return

    if interactive:
        fig = px.scatter(
            df, x="LONGITUDE", y=depth_col, color=var,
            color_continuous_scale="Viridis", height=600,
            title=f"{var} vs {depth_col}"
        )
        fig.update_yaxes(autorange="reversed")  # depth increases downward
        fig.show()
    else:
        import matplotlib.pyplot as plt
        plt.scatter(df["LONGITUDE"], df[depth_col], c=df[var], cmap="viridis", s=2)
        plt.gca().invert_yaxis()
        plt.xlabel("Longitude")
        plt.ylabel(depth_col)
        plt.title(f"{var} vs {depth_col}")
        plt.colorbar(label=var)
        plt.show()

# --- STEP 4: FUNCTION TO COMPUTE REGIONAL SUMMARY ---
def compute_summary(df, depth_interval=100):
    """
    Compute mean TEMP and PSAL by depth intervals.
    """
    if df.empty:
        return pd.DataFrame()

    df["DEPTH_BIN"] = (df["DEPTH_M"] // depth_interval) * depth_interval
    summary = df.groupby("DEPTH_BIN")[["TEMP", "PSAL"]].mean().reset_index()
    return summary

# --- STEP 5: EXAMPLE RUN FOR ALL YEARS ---
for file_name in filtered_files:
    file_path = os.path.join(filtered_dir, file_name)
    print(f"\n🔹 Processing file: {file_name}")

    df = load_and_standardize(file_path)
    print("✅ Loaded dataframe shape:", df.shape)

    # Plot TEMP vs DEPTH_M
    plot_variable(df, var="TEMP", depth_col="DEPTH_M", interactive=True)

    # Compute summary statistics
    summary_df = compute_summary(df, depth_interval=100)
    print("📊 Summary statistics by depth (first 10 rows):")
    print(summary_df.head(10))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.5 MB/s eta 0:00:00
📂 Filtered files available: ['argo_2023_filtered.csv', 'argo_2024_filtered.csv', 'argo_2022_filtered.csv']

🔹 Processing file: argo_2023_filtered.csv
✅ Loaded dataframe shape: (6561601, 7)
Buffered data was truncated after reaching the output size limit.

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install pandas xarray pyarrow fastparquet plotly --quiet
# Downgrade Plotly to version 5.13.1 (works with pandas < 2.2 and pyarrow < 20)
!pip install plotly==5.13.1 --quiet



import os
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --- STEP 1: SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")

# Check available filtered files
filtered_files = [f for f in os.listdir(filtered_dir) if f.endswith(".csv")]
print("📂 Filtered files available:", filtered_files)

# --- STEP 2: LOAD AND STANDARDIZE DATA (memory-safe) ---
def load_and_standardize(file_path):
    """Load CSV, standardize columns, add missing columns, extract YEAR"""
    df = pd.read_csv(file_path)
    rename_map = {
        "LAT": "LATITUDE",
        "LON": "LONGITUDE",
        "TEMP_ADJUSTED": "TEMP",
        "DEPTH": "DEPTH_M"
    }
    df.rename(columns=rename_map, inplace=True)

    required_cols = ["LATITUDE", "LONGITUDE", "JULD", "DEPTH_M", "TEMP", "PSAL"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = pd.NA

    # Extract year from filename
    year_match = re.search(r'(\d{4})', file_path)
    df["YEAR"] = int(year_match.group(1)) if year_match else pd.NA
    df["source_file"] = os.path.basename(file_path)

    return df[required_cols + ["source_file", "YEAR"]]

# --- STEP 3: COMPUTE SUMMARY STATISTICS BY YEAR & DEPTH BIN ---
def compute_summary_by_year(file_path, depth_interval=100):
    """Compute mean TEMP and PSAL per YEAR and DEPTH_BIN for a single file"""
    df = load_and_standardize(file_path)
    df["DEPTH_BIN"] = (df["DEPTH_M"] // depth_interval) * depth_interval
    summary = df.groupby(["YEAR", "DEPTH_BIN"])[["TEMP", "PSAL"]].mean().reset_index()
    return summary

# --- STEP 4: COMBINE SUMMARIES FROM ALL FILES ---
all_summaries = []
for file_name in filtered_files:
    file_path = os.path.join(filtered_dir, file_name)
    print(f"🔹 Processing {file_name} ...")
    summary = compute_summary_by_year(file_path, depth_interval=100)
    all_summaries.append(summary)

summary_df = pd.concat(all_summaries, ignore_index=True)
print("✅ Combined summary shape:", summary_df.shape)
print(summary_df.head(20))

# Optional: Save summary to CSV
summary_path = os.path.join(drive_base, "argo_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"📄 Summary saved to {summary_path}")

# --- STEP 5: PLOT MULTI-YEAR VARIABLES USING SUMMARY ---
def plot_summary_variable(summary_df, var="TEMP"):
    fig = go.Figure()
    for year in sorted(summary_df["YEAR"].dropna().unique()):
        year_data = summary_df[summary_df["YEAR"] == year]
        fig.add_trace(go.Scatter(
            x=year_data[var],
            y=year_data["DEPTH_BIN"],
            mode="lines+markers",
            name=str(year)
        ))
    fig.update_yaxes(autorange="reversed", title_text="Depth (m)")
    fig.update_xaxes(title_text=var)
    fig.update_layout(title=f"Average {var} by Depth Bin")
    fig.show()

# Plot TEMP
plot_summary_variable(summary_df, var="TEMP")
# Plot PSAL
plot_summary_variable(summary_df, var="PSAL")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 96.1 MB/s eta 0:00:00
📂 Filtered files available: ['argo_2023_filtered.csv', 'argo_2024_filtered.csv', 'argo_2022_filtered.csv']
🔹 Processing argo_2023_filtered.csv ...
🔹 Processing argo_2024_filtered.csv ...
🔹 Processing argo_2022_filtered.csv ...
✅ Combined summary shape: (161, 4)
    YEAR  DEPTH_BIN       TEMP       PSAL
0   2025     -100.0  26.961000   0.385667
1   2025        0.0  23.009933  34.516172
2   2025      100.0  17.747541  34.576182
3   2025      200.0  14.800311  34.544905
4   2025      300.0  12.936102  34.423265
5   2025      400.0  11.462688  34.224181
6   2025      500.0  10.522801  34.150085
7   2025      600.0   9.637187  34.077035
8   2025      700.0   8.684264  33.997556
9   2025      800.0   7.513072  33.809641
10  2025      900.0   6.579412  33.798268
11  2025     1000.0   5.813591  34.139474
12  2025     1100.0   5.207584  34.183518
13  2025     1200.0   4.614597  34.238429
14  2025     1300.0   4.20535

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install xarray pandas pyarrow fastparquet plotly sentence-transformers chromadb

import os
import pandas as pd
import xarray as xr
import plotly.express as px
from sentence_transformers import SentenceTransformer
import chromadb

# --- STEP 1: SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
os.makedirs(filtered_dir, exist_ok=True)

# --- STEP 2: LOAD CATALOG ---
catalog_path = os.path.join(drive_base, "argo_metadata_catalog.csv")
catalog = pd.read_csv(catalog_path)
print("📑 Catalog loaded:", catalog.shape)

# --- STEP 3: INITIALIZE EMBEDDING MODEL & VECTOR STORE ---
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection(name="argo_files")

# --- STEP 4: PREPARE METADATA EMBEDDINGS ---
docs, metas = [], []
for idx, row in catalog.iterrows():
    text = (
        f"File {row['file_path']} contains {row['num_rows']} measurements "
        f"from {row['year']}-{row['month']} "
        f"in region lat[{row['lat_min']}, {row['lat_max']}] "
        f"lon[{row['lon_min']}, {row['lon_max']}] "
        f"with depth range {row['depth_min']}–{row['depth_max']} meters."
    )
    docs.append(text)
    metas.append(row.to_dict())

# Generate embeddings
print("🔹 Generating embeddings...")
embeddings = embedder.encode(docs).tolist()

# Insert into ChromaDB
try:
    collection.add(
        documents=docs,
        embeddings=embeddings,
        metadatas=metas,
        ids=[str(i) for i in range(len(docs))]
    )
    print(f"✅ Inserted {len(docs)} entries into vector store.")
except Exception as e:
    print("⚠️ Possibly already added:", e)

# --- STEP 5: HYBRID RETRIEVAL FUNCTION ---
def hybrid_retrieve(query_text, year=None, lat_min=None, lat_max=None,
                    lon_min=None, lon_max=None, depth_min=None, depth_max=None, top_k=5):
    query_emb = embedder.encode([query_text]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=top_k)
    matched_files = pd.DataFrame(results['metadatas'][0])

    # Apply metadata filters
    if year is not None:
        matched_files = matched_files[matched_files['year'] == year]
    if lat_min is not None:
        matched_files = matched_files[matched_files['lat_max'] >= lat_min]
    if lat_max is not None:
        matched_files = matched_files[matched_files['lat_min'] <= lat_max]
    if lon_min is not None:
        matched_files = matched_files[matched_files['lon_max'] >= lon_min]
    if lon_max is not None:
        matched_files = matched_files[matched_files['lon_min'] <= lon_max]
    if depth_min is not None:
        matched_files = matched_files[matched_files['depth_max'] >= depth_min]
    if depth_max is not None:
        matched_files = matched_files[matched_files['depth_min'] <= depth_max]

    return matched_files

# --- STEP 6: LAZY FILE LOADER ---
def load_file_lazy(file_path):
    try:
        if file_path.endswith(".nc"):
            ds = xr.open_dataset(file_path, decode_times=False)
            df = ds.to_dataframe().reset_index()
        elif file_path.endswith(".parquet"):
            df = pd.read_parquet(file_path)
        else:
            return pd.DataFrame()
        df["source_file"] = file_path
        return df
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return pd.DataFrame()

# --- STEP 7: GET DATA FOR QUERY ---
def get_data_for_query(query_text, year=None, lat_min=None, lat_max=None,
                       lon_min=None, lon_max=None, depth_min=None, depth_max=None):
    files = hybrid_retrieve(query_text, year, lat_min, lat_max, lon_min, lon_max, depth_min, depth_max)
    if files.empty:
        print("⚠️ No files found for this query")
        return pd.DataFrame()

    print(f"📂 Loading {len(files)} matching files...")
    all_data = []
    for f in files['file_path']:
        if os.path.exists(f):
            df = load_file_lazy(f)
            if not df.empty:
                all_data.append(df)
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        return final_df
    else:
        print("⚠️ No data loaded from matching files")
        return pd.DataFrame()

# --- STEP 8: PLOTTING FUNCTION ---
def plot_variable(df, var="TEMP", depth_col="DEPTH_M", interactive=True):
    if df.empty or var not in df.columns or depth_col not in df.columns:
        print(f"⚠️ Columns {var} or {depth_col} not found")
        return
    if interactive:
        fig = px.scatter(df, x="LONGITUDE", y=depth_col, color=var,
                         color_continuous_scale="Viridis", height=600,
                         title=f"{var} vs {depth_col}")
        fig.update_yaxes(autorange="reversed")
        fig.show()
    else:
        import matplotlib.pyplot as plt
        plt.scatter(df["LONGITUDE"], df[depth_col], c=df[var], cmap="viridis", s=2)
        plt.gca().invert_yaxis()
        plt.xlabel("Longitude")
        plt.ylabel(depth_col)
        plt.title(f"{var} vs {depth_col}")
        plt.colorbar(label=var)
        plt.show()

# --- STEP 9: EXAMPLE QUERY & PLOT ---
query_text = "temperature profiles"
df2022 = get_data_for_query(query_text, year=2022, lat_min=-20, lat_max=20, lon_min=30, lon_max=60, depth_min=0, depth_max=2000)

if not df2022.empty:
    print("✅ Data loaded. Shape:", df2022.shape)
    # Save merged file to Drive
    df2022.to_csv(os.path.join(filtered_dir, "argo_2022_filtered.csv"), index=False)
    df2022.to_parquet(os.path.join(filtered_dir, "argo_2022_filtered.parquet"), index=False)

    # Plot TEMP vs DEPTH
    plot_variable(df2022, var="TEMP", depth_col="DEPTH_M", interactive=True)

# --- Optional: Repeat for 2023/2024 by changing year parameter ---


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install xarray pandas pyarrow fastparquet plotly matplotlib

import os
import pandas as pd
import xarray as xr
import plotly.express as px
import matplotlib.pyplot as plt

# --- STEP 1: SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
plots_dir = os.path.join(filtered_dir, "plots")
summary_dir = os.path.join(filtered_dir, "summary")
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(summary_dir, exist_ok=True)

# --- STEP 2: HELPER FUNCTIONS ---

def load_filtered_data(file_path):
    """Load filtered CSV or Parquet file"""
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        return pd.DataFrame()

    try:
        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_path.endswith(".parquet"):
            df = pd.read_parquet(file_path)
        else:
            print(f"⚠️ Unsupported file type: {file_path}")
            return pd.DataFrame()

        # Standardize column names
        df = df.rename(columns={
            "DEPTH_M": "DEPTH",
            "PRES": "DEPTH",
            "TEMP_ADJUSTED": "TEMP",
            "TEMP": "TEMP",
            "PSAL": "PSAL"
        })

        # Keep only relevant columns
        columns_needed = ["LATITUDE", "LONGITUDE", "DEPTH", "TEMP", "PSAL", "source_file"]
        for col in columns_needed:
            if col not in df.columns:
                df[col] = pd.NA  # Fill missing columns

        return df[columns_needed]

    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return pd.DataFrame()


def plot_variable(df, var="TEMP", interactive=True, year=None):
    """Plot variable vs depth"""
    if df.empty or var not in df.columns or "DEPTH" not in df.columns:
        print(f"⚠️ Columns {var} or DEPTH not found in dataframe")
        return

    title = f"{var} vs Depth"
    if year:
        title += f" ({year})"

    if interactive:
        fig = px.scatter(df, x="LONGITUDE", y="DEPTH", color=var,
                         color_continuous_scale="Viridis", height=600, title=title)
        fig.update_yaxes(autorange="reversed")
        file_html = os.path.join(plots_dir, f"{var}_{year}.html")
        fig.write_html(file_html)
        print(f"💾 Saved interactive plot: {file_html}")
    else:
        plt.figure(figsize=(10,6))
        plt.scatter(df["LONGITUDE"], df["DEPTH"], c=df[var], cmap="viridis", s=2)
        plt.gca().invert_yaxis()
        plt.xlabel("Longitude")
        plt.ylabel("Depth (m)")
        plt.title(title)
        file_png = os.path.join(plots_dir, f"{var}_{year}.png")
        plt.colorbar(label=var)
        plt.savefig(file_png)
        plt.close()
        print(f"💾 Saved static plot: {file_png}")


def summarize_data(df, year=None):
    """Compute summary statistics"""
    if df.empty:
        return pd.DataFrame()

    summary = df[["DEPTH", "TEMP", "PSAL"]].agg(["count", "mean", "std", "min", "max"])
    file_summary = os.path.join(summary_dir, f"summary_{year}.csv")
    summary.to_csv(file_summary)
    print(f"💾 Saved summary CSV: {file_summary}")
    return summary

# --- STEP 3: PROCESS ALL YEARS ---
years = [2022, 2023, 2024]  # Add more if available
for year in years:
    print(f"\n🔹 Processing year {year} ...")
    file_csv = os.path.join(filtered_dir, f"argo_{year}_filtered.csv")

    df_year = load_filtered_data(file_csv)
    if df_year.empty:
        print(f"⚠️ No data found for year {year}")
        continue

    # Plot TEMP and PSAL
    plot_variable(df_year, var="TEMP", interactive=True, year=year)
    plot_variable(df_year, var="PSAL", interactive=True, year=year)

    # Generate summary statistics
    summarize_data(df_year, year=year)

print("\n✅ All years processed. Plots and summaries saved to Google Drive.")



🔹 Processing year 2022 ...
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plots/TEMP_2022.html
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plots/PSAL_2022.html
💾 Saved summary CSV: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/summary/summary_2022.csv

🔹 Processing year 2023 ...
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plots/TEMP_2023.html
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plots/PSAL_2023.html
💾 Saved summary CSV: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/summary/summary_2023.csv

🔹 Processing year 2024 ...
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plots/TEMP_2024.html
💾 Saved interactive plot: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/filtered_argo_data/plo

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install -U plotly pandas xarray pyarrow fastparquet --quiet

import os
import pandas as pd
import plotly.express as px

# --- SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
heatmap_dir = os.path.join(drive_base, "heatmaps")
os.makedirs(heatmap_dir, exist_ok=True)

# --- LIST FILTERED FILES ---
filtered_files = [f for f in os.listdir(filtered_dir) if f.endswith(".parquet")]

# --- FUNCTION TO COMPUTE SUMMARY ---
def summarize_for_heatmap(df, variable, depth_interval=100, time_bin='M'):
    """
    Summarize the dataframe for faster heatmap plotting:
    - Bin depths by `depth_interval`
    - Bin times by month (or as needed)
    - Compute mean value of `variable` per bin
    """
    df = df.copy()
    if 'DEPTH_M' not in df.columns or variable not in df.columns or 'JULD' not in df.columns:
        return pd.DataFrame()  # empty

    # Create depth bins
    df['DEPTH_BIN'] = (df['DEPTH_M'] // depth_interval) * depth_interval

    # Convert JULD to datetime if not already
    if not pd.api.types.is_datetime64_any_dtype(df['JULD']):
        df['JULD'] = pd.to_datetime(df['JULD'], errors='coerce')

    # Create time bins
    df['TIME_BIN'] = df['JULD'].dt.to_period(time_bin).dt.to_timestamp()

    # Group and compute mean
    summary = df.groupby(['TIME_BIN', 'DEPTH_BIN'])[variable].mean().reset_index()
    return summary

# --- FUNCTION TO PLOT HEATMAP (HTML only) ---
def plot_depth_heatmap_summary(summary_df, variable, year):
    if summary_df.empty:
        print(f"⚠️ No data to plot for {variable} ({year})")
        return

    fig = px.density_heatmap(
        summary_df,
        x='TIME_BIN',
        y='DEPTH_BIN',
        z=variable,
        nbinsx=200,
        nbinsy=50,
        color_continuous_scale='Viridis',
        title=f"{variable} vs Depth over Time - {year}"
    )
    fig.update_yaxes(autorange='reversed', title='Depth (m)')
    fig.update_xaxes(title='Time')

    # Save as interactive HTML only
    html_path = os.path.join(heatmap_dir, f"{variable}_heatmap_{year}.html")
    fig.write_html(html_path)
    print(f"💾 Saved interactive heatmap for {variable} ({year}) -> {html_path}")

# --- PROCESS EACH YEAR ---
for file in filtered_files:
    year = file.split("_")[1]  # assuming filename: argo_2022_filtered.parquet
    file_path = os.path.join(filtered_dir, file)

    print(f"\n🔹 Processing year {year} ...")
    df = pd.read_parquet(file_path)

    # TEMP summary + heatmap
    temp_summary = summarize_for_heatmap(df, variable='TEMP', depth_interval=100)
    plot_depth_heatmap_summary(temp_summary, variable='TEMP', year=year)

    # PSAL summary + heatmap
    psal_summary = summarize_for_heatmap(df, variable='PSAL', depth_interval=100)
    plot_depth_heatmap_summary(psal_summary, variable='PSAL', year=year)



🔹 Processing year 2023 ...
💾 Saved interactive heatmap for TEMP (2023) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/TEMP_heatmap_2023.html
💾 Saved interactive heatmap for PSAL (2023) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/PSAL_heatmap_2023.html

🔹 Processing year 2024 ...
💾 Saved interactive heatmap for TEMP (2024) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/TEMP_heatmap_2024.html
💾 Saved interactive heatmap for PSAL (2024) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/PSAL_heatmap_2024.html

🔹 Processing year 2022 ...
💾 Saved interactive heatmap for TEMP (2022) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/TEMP_heatmap_2022.html
💾 Saved interactive heatmap for PSAL (2022) -> /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/heatmaps/PSAL_heatmap_2022.html


In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install -U plotly pandas xarray pyarrow fastparquet kaleido --quiet

# --- INSTALL HEADLESS CHROME FOR KALEIDO ---
!apt-get update -qq
!apt-get install -y chromium-browser

import os
import pandas as pd
import xarray as xr
import plotly.express as px
import kaleido

# --- SETUP PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
plots_dir = os.path.join(drive_base, "argo_heatmaps")
os.makedirs(plots_dir, exist_ok=True)

# --- LAZY FILE LOADER / SUMMARY ---
def load_and_summarize(file_path, sample_frac=0.05, depth_interval=100):
    """
    Load CSV/Parquet/NetCDF and summarize data:
    - Sample fraction for memory safety
    - Create depth bins for heatmap plotting
    """
    try:
        if file_path.endswith(".nc"):
            ds = xr.open_dataset(file_path, decode_times=False)
            df = ds.to_dataframe().reset_index()
        elif file_path.endswith(".parquet"):
            df = pd.read_parquet(file_path)
        elif file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        else:
            return pd.DataFrame()

        df["source_file"] = file_path

        # Sample data if too large
        if len(df) > 50000:
            df = df.sample(n=50000, random_state=42)

        # Depth bins
        if "DEPTH_M" in df.columns:
            df["DEPTH_BIN"] = (df["DEPTH_M"] // depth_interval) * depth_interval
        else:
            df["DEPTH_BIN"] = pd.NA

        return df
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return pd.DataFrame()

# --- PLOT FUNCTION (HTML + optional PNG) ---
def plot_depth_heatmap(df, variable="TEMP", depth_col="DEPTH_BIN", year=2022, save_png=False):
    if df.empty or variable not in df.columns or depth_col not in df.columns:
        print(f"⚠️ Columns {variable} or {depth_col} not found in dataframe")
        return

    fig = px.scatter(
        df, x="LONGITUDE", y=depth_col, color=variable,
        color_continuous_scale="Viridis", height=600,
        title=f"{variable} vs {depth_col} ({year})"
    )
    fig.update_yaxes(autorange="reversed")

    # Save interactive HTML
    html_path = os.path.join(plots_dir, f"{variable}_{year}.html")
    fig.write_html(html_path)
    print(f"💾 Saved HTML heatmap: {html_path}")

    # Save PNG if requested (requires Chrome)
    if save_png:
        try:
            png_path = os.path.join(plots_dir, f"{variable}_{year}.png")
            fig.write_image(png_path)
            print(f"💾 Saved PNG heatmap: {png_path}")
        except Exception as e:
            print(f"⚠️ Could not save PNG: {e}")

# --- PROCESS ALL YEARS ---
years = [2022, 2023, 2024]  # adjust as needed

for year in years:
    print(f"\n🔹 Processing year {year} ...")
    csv_file = os.path.join(filtered_dir, f"argo_{year}_filtered.csv")

    if not os.path.exists(csv_file):
        print(f"⚠️ File not found: {csv_file}")
        continue

    df = load_and_summarize(csv_file, sample_frac=0.1)
    print(f"📄 Loaded {df.shape[0]} rows for {year}")

    # Plot TEMP
    plot_depth_heatmap(df, variable="TEMP", depth_col="DEPTH_BIN", year=year, save_png=False)

    # Plot PSAL
    plot_depth_heatmap(df, variable="PSAL", depth_col="DEPTH_BIN", year=year, save_png=False)

print("\n✅ All heatmaps generated and saved.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  apparmor libfuse3-3 libudev1 snapd squashfs-tools systemd-hwe-hwdb udev
Suggested packages:
  apparmor-profiles-extra apparmor-utils fuse3 zenity | kdialog
The following NEW packages will be installed:
  apparmor chromium-browser libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb
  udev
The following packages will be upgraded:
  libudev1
1 upgraded, 7 newly installed, 0 to remove and 45 not upgraded.
Need to get 32.5 MB of archives.
After this operation, 130 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 apparmor amd64 3.0.4-2ubuntu2.4 [598 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main a

In [ ]:
import pandas as pd
import os

# Paths
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
summary_dir = os.path.join(drive_base, "argo_summaries")
os.makedirs(summary_dir, exist_ok=True)

# Years to process
years = [2022, 2023, 2024]

# Initialize combined summary
all_summary = []

for year in years:
    csv_file = os.path.join(filtered_dir, f"argo_{year}_filtered.csv")
    if not os.path.exists(csv_file):
        print(f"⚠️ File not found: {csv_file}")
        continue

    df = pd.read_csv(csv_file)

    # Depth bins
    df["DEPTH_BIN"] = (df["DEPTH_M"] // 100) * 100

    # Compute mean TEMP and PSAL per depth bin
    summary = df.groupby("DEPTH_BIN")[["TEMP", "PSAL"]].mean().reset_index()
    summary["YEAR"] = year

    all_summary.append(summary)

combined_summary = pd.concat(all_summary, ignore_index=True)
print("✅ Combined summary shape:", combined_summary.shape)
combined_summary.head()


✅ Combined summary shape: (161, 4)


,DEPTH_BIN,TEMP,PSAL,YEAR
0,-500.0,61.166000,61.166000,2022
1,-100.0,33.533333,0.035667,2022
2,0.0,22.707559,34.623503,2022
3,100.0,17.898483,34.721525,2022
4,200.0,15.053747,34.738191,2022


from matplotlib import pyplot as plt
_df_0['DEPTH_BIN'].plot(kind='hist', bins=20, title='DEPTH_BIN')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['TEMP'].plot(kind='hist', bins=20, title='TEMP')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['PSAL'].plot(kind='hist', bins=20, title='PSAL')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['DEPTH_BIN']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'DEPTH_BIN'}, axis=1)
              .sort_values('DEPTH_BIN', ascending=True))
  xs = counted['DEPTH_BIN']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_7.sort_values('DEPTH_BIN', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('DEPTH_BIN')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['YEAR']
  ys = series['TEMP']
  
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_8.sort_values('YEAR', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('YEAR')
_ = plt.ylabel('TEMP')

from matplotlib import pyplot as plt
_df_9['DEPTH_BIN'].plot(kind='line', figsize=(8, 4), title='DEPTH_BIN')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_10['TEMP'].plot(kind='line', figsize=(8, 4), title='TEMP')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_11['PSAL'].plot(kind='line', figsize=(8, 4), title='PSAL')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
import plotly.express as px

def plot_multi_year_summary(df, variable="TEMP"):
    fig = px.scatter(
        df,
        x="DEPTH_BIN",
        y=variable,
        color="YEAR",
        facet_col="YEAR",
        color_continuous_scale="Viridis",
        title=f"{variable} Profile by Depth and Year",
        height=500
    )
    fig.update_yaxes(autorange="reversed")  # depth increases downward
    return fig

# TEMP plot
fig_temp = plot_multi_year_summary(combined_summary, variable="TEMP")
fig_temp.show()

# PSAL plot
fig_psal = plot_multi_year_summary(combined_summary, variable="PSAL")
fig_psal.show()


In [ ]:
def plot_smoothed_profiles(df, variable="TEMP"):
    import plotly.graph_objects as go

    fig = go.Figure()

    for year in sorted(df["YEAR"].unique()):
        df_year = df[df["YEAR"]==year]
        fig.add_trace(go.Scatter(
            x=df_year[variable],
            y=df_year["DEPTH_BIN"],
            mode="lines+markers",
            name=str(year)
        ))

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"Smoothed {variable} Profile by Depth",
        xaxis_title=variable,
        yaxis_title="DEPTH_BIN (m)",
        height=600
    )
    fig.show()

# TEMP profile
plot_smoothed_profiles(combined_summary, "TEMP")

# PSAL profile
plot_smoothed_profiles(combined_summary, "PSAL")


In [ ]:
import os
import pandas as pd
import numpy as np

# --- PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
filtered_dir = os.path.join(drive_base, "filtered_argo_data")
summary_dir = os.path.join(drive_base, "argo_summary")
os.makedirs(summary_dir, exist_ok=True)

# --- YEARS ---
years = [2022, 2023, 2024]

# --- FUNCTION TO BIN DEPTHS ---
def depth_bins(df, depth_col="DEPTH_M", bin_size=50):
    """Add depth_bin column for aggregation"""
    df["depth_bin"] = (df[depth_col] // bin_size) * bin_size
    return df

# --- PROCESS EACH YEAR ---
for year in years:
    csv_file = os.path.join(filtered_dir, f"argo_{year}_filtered.csv")

    if not os.path.exists(csv_file):
        print(f"⚠️ File not found: {csv_file}")
        continue

    df = pd.read_csv(csv_file)
    print(f"\n🔹 Processing year {year}, {df.shape[0]} rows loaded")

    # Bin depths
    df = depth_bins(df, depth_col="DEPTH_M", bin_size=50)

    # Compute summary stats per depth_bin
    summary = df.groupby("depth_bin").agg(
        TEMP_mean=("TEMP", "mean"),
        TEMP_min=("TEMP", "min"),
        TEMP_max=("TEMP", "max"),
        TEMP_std=("TEMP", "std"),
        PSAL_mean=("PSAL", "mean"),
        PSAL_min=("PSAL", "min"),
        PSAL_max=("PSAL", "max"),
        PSAL_std=("PSAL", "std"),
    ).reset_index()

    # Save summary CSV
    summary_file = os.path.join(summary_dir, f"argo_{year}_summary.csv")
    summary.to_csv(summary_file, index=False)
    print(f"💾 Summary saved: {summary_file}")



🔹 Processing year 2022, 4746064 rows loaded
💾 Summary saved: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/argo_summary/argo_2022_summary.csv

🔹 Processing year 2023, 6561601 rows loaded
💾 Summary saved: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/argo_summary/argo_2023_summary.csv

🔹 Processing year 2024, 1779386 rows loaded
💾 Summary saved: /content/drive/MyDrive/ColabNotebooks/SIH2025/Data/argo_summary/argo_2024_summary.csv


In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install -q dash plotly pandas

import os
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# --- PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
summary_dir = os.path.join(drive_base, "argo_summary")  # summarized CSVs

# --- AVAILABLE YEARS & VARIABLES ---
years = [2022, 2023, 2024]
variables = ["TEMP", "PSAL"]

# --- INITIALIZE DASH APP ---
app = Dash(__name__)
app.title = "ARGO Data Dashboard"

# --- DASH LAYOUT ---
app.layout = html.Div([
    html.H1("ARGO Ocean Profile Dashboard", style={"textAlign": "center"}),

    html.Div([
        html.Label("Select Year:"),
        dcc.Dropdown(
            id="year-dropdown",
            options=[{"label": y, "value": y} for y in years],
            value=2022,
            clearable=False
        ),
    ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

    html.Div([
        html.Label("Select Variable:"),
        dcc.Dropdown(
            id="var-dropdown",
            options=[{"label": v, "value": v} for v in variables],
            value="TEMP",
            clearable=False
        ),
    ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

    html.Div([
        html.Label("Depth Range (m):"),
        dcc.RangeSlider(
            id="depth-slider",
            min=0,
            max=2000,
            step=50,
            value=[0, 2000],
            marks={i: str(i) for i in range(0, 2001, 200)}
        )
    ], style={"width": "80%", "padding": "20px"}),

    dcc.Graph(id="heatmap-graph")
])

# --- CALLBACK TO UPDATE HEATMAP ---
@app.callback(
    Output("heatmap-graph", "figure"),
    Input("year-dropdown", "value"),
    Input("var-dropdown", "value"),
    Input("depth-slider", "value")
)
def update_heatmap(selected_year, selected_var, depth_range):
    # Load summarized CSV
    summary_file = os.path.join(summary_dir, f"argo_{selected_year}_summary.csv")
    if not os.path.exists(summary_file):
        return px.scatter(title="Data not found for selected year")

    df = pd.read_csv(summary_file)

    # Ensure column names match summarized CSV
    mean_col = f"{selected_var}_mean"
    std_col = f"{selected_var}_std"
    if mean_col not in df.columns or std_col not in df.columns:
        return px.scatter(title=f"Variable {selected_var} not found")

    # Filter by depth range
    df = df[(df["depth_bin"] >= depth_range[0]) & (df["depth_bin"] <= depth_range[1])]

    # Line plot with error bars
    fig = px.line(
        df,
        x="depth_bin",
        y=mean_col,
        error_y=std_col,
        labels={"depth_bin": "Depth (m)", mean_col: f"{selected_var} Mean"},
        title=f"{selected_var} Profile - {selected_year}"
    )
    fig.update_yaxes(autorange="reversed")  # Depth increases downward
    fig.update_layout(height=600)

    return fig

# --- RUN DASH APP INLINE IN NOTEBOOK ---
app.run(mode="inline", debug=True)


<IPython.core.display.Javascript object>

In [ ]:
# --- INSTALL DEPENDENCIES ---
!pip install dash plotly pandas

import os
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# --- PATHS ---
drive_base = "/content/drive/MyDrive/ColabNotebooks/SIH2025/Data"
summary_dir = os.path.join(drive_base, "argo_summary")

# --- AVAILABLE YEARS & VARIABLES ---
years = [2022, 2023, 2024]
variables = ["TEMP", "PSAL"]

# --- INITIALIZE DASH APP ---
app = Dash(__name__)
app.title = "ARGO Ocean 2D Heatmap Dashboard"

# --- DASH LAYOUT ---
app.layout = html.Div([
    html.H1("ARGO Ocean Profiles 2D Heatmap", style={"textAlign": "center"}),

    html.Div([
        html.Label("Select Year:"),
        dcc.Dropdown(id="year-dropdown", options=[{"label": y, "value": y} for y in years],
                     value=2022, clearable=False),
    ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

    html.Div([
        html.Label("Select Variable:"),
        dcc.Dropdown(id="var-dropdown", options=[{"label": v, "value": v} for v in variables],
                     value="TEMP", clearable=False),
    ], style={"width": "30%", "display": "inline-block", "padding": "10px"}),

    html.Div([
        html.Label("Depth Range (m):"),
        dcc.RangeSlider(id="depth-slider", min=0, max=2000, step=50, value=[0, 2000],
                        marks={i: str(i) for i in range(0, 2001, 200)})
    ], style={"width": "80%", "padding": "20px"}),

    dcc.Graph(id="heatmap-graph")
])

# --- CALLBACK TO UPDATE HEATMAP ---
@app.callback(
    Output("heatmap-graph", "figure"),
    Input("year-dropdown", "value"),
    Input("var-dropdown", "value"),
    Input("depth-slider", "value")
)
def update_heatmap(selected_year, selected_var, depth_range):
    # Load summarized CSV for the selected year
    summary_file = os.path.join(summary_dir, f"argo_{selected_year}_summary.csv")
    if not os.path.exists(summary_file):
        return px.scatter(title="Data not found for selected year")

    df = pd.read_csv(summary_file)

    # Filter by depth range
    df = df[(df["depth_bin"] >= depth_range[0]) & (df["depth_bin"] <= depth_range[1])]

    if df.empty:
        return px.scatter(title="No data in selected depth range")

    # Create 2D heatmap: Depth vs Longitude
    fig = px.density_heatmap(
        df, x="LONGITUDE", y="depth_bin", z=f"{selected_var}_mean",
        histfunc="avg", color_continuous_scale="Viridis",
        labels={"depth_bin": "Depth (m)", "LONGITUDE": "Longitude", f"{selected_var}_mean": f"{selected_var} Mean"},
        title=f"{selected_var} Heatmap - {selected_year}"
    )
    fig.update_yaxes(autorange="reversed")  # Depth increases downward
    fig.update_layout(height=700)
    return fig

# --- RUN DASH APP ---
if __name__ == "__main__":
    app.run_server(debug=True, port=8050)


ObsoleteAttributeException: app.run_server has been replaced by app.run